In [5]:
import pandas as pd
import numpy as np
from datetime import timedelta
import random

## This script generates synthetic data for a hypothetical e-commerce platform.
df = pd.read_csv(r'C:\Users\neilm\Downloads\sales_details.csv')
# DATA QUALITY CHECK Identify any values in sls_order_dt that cannot be parsedas YYYYMMDD dates. 
# These will be safely handled later.
invalid_dates = pd.to_datetime(df['sls_order_dt'], format='%Y%m%d', errors='coerce').isna()
print("Invalid date values detected:", df.loc[invalid_dates, 'sls_order_dt'].unique())
print("Number of invalid dates:", invalid_dates.sum())

# Target dataset size
target_rows = 200000
rows_to_generate = target_rows - len(df)

#Extract valid keys to maintain referential integrity (ensures fact table still matches dimension tables)
product_keys = df['sls_prd_key'].dropna().unique()
customer_ids = df['sls_cust_id'].dropna().unique()

#Build product base price lookup using median price
#This prevents extreme values influencing generated prices
product_base_prices = df.groupby('sls_prd_key')['sls_price'].median().to_dict()
#Base year used for price inflation/deflation logic
base_year = 2014
# Determine starting point for sequential order numbers
max_order = (
    df['sls_ord_num']
    .astype(str)
    .str.replace('SO', '', regex=False)
    .astype(int)
    .max()
)
# Container for generated rows
new_rows = []
#-----------------------------------------------------------
#Generate synthetic sales rows
#-----------------------------------------------------------
for i in range(rows_to_generate):
    # Sample an existing order date and shift forward
    sample_value = df['sls_order_dt'].sample().values[0]
    try:
        order_date = pd.to_datetime(sample_value, format='%Y%m%d') + timedelta(days=random.randint(1,365))
    except:
        # Skip if sampled value is not a valid date
        continue
    ship_date = order_date + timedelta(days=random.randint(1,10))
    due_date = ship_date + timedelta(days=random.randint(1,5))
    # Select product
    product = random.choice(product_keys)
    # Product + Year pricing model
    # Earlier years cheaper, later years slightly higher
    base_price = product_base_prices.get(product, 100)
    year_diff = order_date.year - base_year
    price = round(base_price * (1.02 ** year_diff), 2)
    # Realistic purchase quantity distribution
    # Most purchases are 1–2 units
    quantity = np.random.choice(
    [1,2,3,4,5],
    p=[0.45,0.30,0.15,0.07,0.03]
    )
    # Revenue calculation
    sales = round(quantity * price, 2)
    # Sequential order numbers
    order_number = f"SO{max_order + i + 1}"
    new_rows.append({
        'sls_ord_num': order_number,
        'sls_prd_key': product,
        'sls_cust_id': random.choice(customer_ids),
        'sls_order_dt': order_date.strftime('%Y%m%d'),
        'sls_ship_dt': ship_date.strftime('%Y%m%d'),
        'sls_due_dt': due_date.strftime('%Y%m%d'),
        'sls_sales': sales,
        'sls_quantity': quantity,
        'sls_price': price
    })
# Create DataFrame from generated rows
new_df = pd.DataFrame(new_rows)
#Introduce small data quality issues (useful for portfolio data cleaning demonstrations)
for idx in new_df.sample(frac=0.05).index:
    # Missing revenue example
    if random.random() < 0.5:
        new_df.at[idx, 'sls_sales'] = np.nan
    # Mistyped product key example
    if random.random() < 0.5:
        new_df.at[idx, 'sls_prd_key'] = 'Mistyped'
# Combine original dataset with generated rows
#-----------------------------------------------------------
final_df = pd.concat([df, new_df], ignore_index=True)
#-----------------------------------------------------------
#Safely derive order year column
errors='coerce' # ensures invalid dates become NaT
final_df['order_year'] = pd.to_datetime(
    final_df['sls_order_dt'], format='%Y%m%d', errors='coerce'
).dt.year
#Export final dataset
final_df.to_csv('expanded_sales_data.csv', index=False)
print("Dataset successfully created with", len(final_df), "rows")

Invalid date values detected: [    0 32154  5489]
Number of invalid dates: 19
Dataset successfully created with 199957 rows
